**Objectif : prédire la région d’une personne à partir de ses préférences musicales.** 

 13 genres musicaux (`Q5_1` à `Q5_13`) et la région (`REG13`)
 Une ligne correspond à une personne

In [95]:
import pandas as pd
import numpy as np

In [96]:
filepath = "../documentation/hadopi-2020-musique-data.ods"
df = pd.read_excel(filepath, engine="odf", na_values=["#NUL!"])
df.shape

(2200, 261)

Parmis celle qu'on a gardé, verifions si plusieurs colonnes donnent la localisation 

Le fichier HADOPI contient plusieurs colonnes qui indiquent où vit la personne, de façon directe ou indirecte :

- `REG` : découpage en 9 zones UDA
- `REG2` : découpage en 5 zones
- `REG3` : Île-de-France / province
- `REG13` et `REG13C` : les 13 régions
- `AGGLOIFOP2` : taille de l'agglomération (nombre d'habitants)

Le problème : si une de ces colonnes se retrouve dans les variables d'entrée, le modèle n'apprend rien sur la musique. Il lit directement la réponse. C'est ce qu'on appelle une **fuite de données**.

Ici, on liste ces colonnes. Puis on cherche par mot-clé (REG, AGGLO, DEP, COMMUNE…) s'il en existe d'autres qu'on n'aurait pas repérées.

In [97]:
# Colonnes qui donnent la localisation, directement ou non
colonnes_localisation = ["REG", "REG2", "REG3", "REG13", "REG13C", "AGGLOIFOP2"]

presentes = [c for c in colonnes_localisation if c in df.columns]
absentes = [c for c in colonnes_localisation if c not in df.columns]
print("Colonnes de localisation présentes :", presentes)
print("Colonnes attendues mais absentes :", absentes)

# Recherche d'autres colonnes suspectes par leur nom
mots_suspects = ["REG", "AGGLO", "DEP", "COMMUNE", "VILLE", "HABITAT", "CP"]
suspectes = [c for c in df.columns
             if any(m in str(c).upper() for m in mots_suspects)
             and c not in colonnes_localisation]
print("Autres colonnes à vérifier à la main :", suspectes)

Colonnes de localisation présentes : ['REG', 'REG2', 'REG3', 'REG13', 'REG13C', 'AGGLOIFOP2']
Colonnes attendues mais absentes : []
Autres colonnes à vérifier à la main : []


- *Colonnes présentes* : elles existent bien dans le fichier, il faudra les écarter.
- *Colonnes absentes* : il s'agit d'une colonne dont le nom attendu ne correspond pas au fichier ici on en a aucune
- *Autres colonnes à vérifier* : la recherche par mot-clé ratisse large. Une colonne peut contenir « DEP » sans avoir de lien avec un département. Chaque nom de la liste doit être contrôlé à la main RAS ici aussi.

Cette cellule prouve notre première selection qui était : (13 styles + REG13) et qui écarte déjà ces colonnes. 

## Garder les colonnes utiles
On écarte les autres colonnes, comme l’âge, le sexe, et les autres informations

La cible est `REG13`

In [98]:
style = [f"Q5_{i}" for i in range(1, 14)]
colonnes_utiles = style + ["REG13"]

dataset = df[colonnes_utiles].copy()

# Convertir le code d’absence avant filtrage
dataset = dataset.replace("#NUL!", pd.NA)
dataset.head()

,Q5_1,Q5_2,Q5_3,Q5_4,Q5_5,Q5_6,Q5_7,Q5_8,Q5_9,Q5_10,Q5_11,Q5_12,Q5_13,REG13
0,Variété Française,0,0,0,Jazz,0,0,0,0,Soul,0,0,0,LanguedicRoussillonMidiP
1,0,Pop / Rock,0,0,Jazz,0,0,0,0,0,0,Musiques du monde,0,PaysDeLaLoire1
2,Variété Française,Pop / Rock,Rap / Hip-Hop,0,0,0,0,0,R&B,0,0,0,0,NordPasDeCalaisPicardie
3,Variété Française,0,0,Classique,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,Variété Française,0,0,0,Jazz,0,0,0,0,0,0,Musiques du monde,0,AuvergneRhoneAlpes


In [99]:
dataset.isna().sum()

Q5_1     214
Q5_2     214
Q5_3     214
Q5_4     214
Q5_5     214
Q5_6     214
Q5_7     214
Q5_8     214
Q5_9     214
Q5_10    214
Q5_11    214
Q5_12    214
Q5_13    214
REG13      0
dtype: int64

## On traite les valeurs manquantes
La ligne `replace("#NUL!", pd.NA)` ==> convertis les absents en vraies valeurs manquantes, même s’ils n’ont pas été reconnus à la lecture du fichier. `dropna()` les retirer

Un `0` reste une bonne réponse 

In [100]:
styles = dataset[style].isna().all(axis=1)
style_incomplets = dataset[style].isna().any(axis=1)

print("Personnes sans réponse musicale :", styles.sum())
print("style partiellement renseignés :", (style_incomplets & ~styles).sum())
print("Région manquante :", dataset["REG13"].isna().sum())

Personnes sans réponse musicale : 214
style partiellement renseignés : 0
Région manquante : 0


Les **214 personnes sans genre renseigné** sont exclues de cette première version sur 2200 

On ne transforme pas leurs absences en `0`, car cela inventerait des réponses.
Il reste donc 1 986 personnes

### supprimer ou remplacer les réponses manquantes

Une fois que nous avons retiré toute personne qui a au moins une réponse manquante. C'est simple et sûr. Mais on peut perdre des répondants qui ont répondu à presque tout.

On compare donc deux méthodes :

- **Suppression** (méthode déjà faite) : une seule case vide, et la ligne est retirée.
- **Remplacement** : si une personne a répondu à certains styles mais pas à d'autres, les styles sans réponse passent à 0 (« n'écoute pas »).

Dans les deux cas, on retire toujours :
- les personnes qui n'ont répondu à **aucun** style (aucune info musicale) ;
- les personnes **sans région** (impossible de savoir quoi prédire).

IMPORTANT : Le remplacement repose sur une hypothèse : « pas de réponse = n'écoute pas ce style ». Elle est discutable, il faut la justifier.

In [101]:
def appliquer_manques(donnees, style, colonnes_utiles, methode="suppression"):
    """
    suppression   : on retire toute ligne avec au moins un manque (méthode de Sara)
    remplacement  : un style non renseigné vaut 0 (non écouté), sauf si la personne
                    n'a répondu à aucun style. Région manquante = ligne retirée.
    """
    donnees = donnees.copy()
    if methode == "remplacement":
        vides = donnees[style].isna().all(axis=1)
        partiels = donnees[style].isna().any(axis=1) & ~vides
        donnees.loc[partiels, style] = donnees.loc[partiels, style].fillna("0")
    return donnees.dropna(subset=colonnes_utiles).reset_index(drop=True)


methode_a = appliquer_manques(dataset, style, colonnes_utiles, "suppression")
methode_b = appliquer_manques(dataset, style, colonnes_utiles, "remplacement")

comparaison = pd.DataFrame({
    "Suppression (%)": (methode_a["REG13"].value_counts(normalize=True) * 100).round(1),
    "Remplacement (%)": (methode_b["REG13"].value_counts(normalize=True) * 100).round(1),
})

print("Personnes conservées - suppression :", len(methode_a))
print("Personnes conservées - remplacement :", len(methode_b))
print("Personnes récupérées :", len(methode_b) - len(methode_a))
display(comparaison)

Personnes conservées - suppression : 1986
Personnes conservées - remplacement : 1986
Personnes récupérées : 0


,Suppression (%),Remplacement (%)
REG13,,
IleDeFrance1,20.3,20.3
AuvergneRhoneAlpes,11.9,11.9
AlsaceChampagneArdenneLo,10.1,10.1
AquitaineLimousinPoitouC,9.8,9.8
NordPasDeCalaisPicardie,9.5,9.5
LanguedicRoussillonMidiP,8.1,8.1
ProvenceAlpesCoteDAzur,6.5,6.5
PaysDeLaLoire1,6.3,6.3
Bretagne1,4.9,4.9


In [102]:
METHODE_MANQUES = "suppression"   # ou "remplacement" selon le résultat

- *Personnes conservées* : nombre de lignes gardées avec chaque méthode.
- *Personnes récupérées* : ce que le remplacement fait gagner par rapport à la suppression.
- *Tableau par région* : part de chaque région (en %) selon la méthode.

Comment décider :

- Si **0 personne récupérée** : les deux méthodes donnent le même résultat. On garde la suppression, plus simple.
- Si des personnes sont récupérées **et que les pourcentages par région bougent peu** (moins d'1 point environ) : le remplacement apporte des données en plus sans déformer la répartition.
- Si les pourcentages **bougent nettement** : le remplacement favorise certaines régions. Mieux vaut garder la suppression.

Résultat obtenu : Nous avons exactement les mêmes valeurs donc décision -> on garde la suppression 



In [103]:
nombre_depart = len(dataset)
dataset = dataset.dropna(subset=colonnes_utiles).copy()
dataset = dataset.reset_index(drop=True)

print("Personnes au départ :", nombre_depart)
print("Personnes retirées :", nombre_depart - len(dataset))
print("Personnes conservées :", len(dataset))

Personnes au départ : 2200
Personnes retirées : 214
Personnes conservées : 1986


## Transformer les préférences en nombres
**1 = sélectionné, 0 = non sélectionné.**

On repart des données brutes pour le codage, 
Les zéros numériques ou textuels (`0`) sont reconnus et les espaces autour des réponses sont retirés.

Si une réponse est inconnue, sa colonne et sa valeur sont affichées => trouver la cause au lieu de remplacer une réponse lambda

Le remplacement de `#NUL!` est aussi fait ici, avant `dropna()`

In [104]:
# Repartir des données brutes, sans réutiliser un codage précédent.
profils_complets = df[colonnes_utiles].copy()
profils_complets = profils_complets.replace("#NUL!", pd.NA)
#profils_complets = profils_complets.dropna(subset=colonnes_utiles).copy()
#----------------------------------- a justifier-----------------------------------------------
profils_complets = appliquer_manques(profils_complets, style, colonnes_utiles, METHODE_MANQUES)
dataset_code = profils_complets.copy()

codage = {
    '0': 0,
    '1': 1,
    'Variété Française': 1,
    'Pop / Rock': 1,
    'Rap / Hip-Hop': 1,
    'Classique': 1,
    'Jazz': 1,
    'Dance': 1,
    'Electronique': 1,
    'Métal': 1,
    'R&B': 1,
    'Soul': 1,
    'Reggae': 1,
    'Musiques du monde': 1,
    'Autres genres': 1,
}

for colonne in style:
    reponses = profils_complets[colonne].astype("string").str.strip()
    inconnues = reponses[~reponses.isin(codage)]

    if not inconnues.empty:
        print("Réponses non reconnues dans", colonne)
        display(inconnues.value_counts(dropna=False))

    dataset_code[colonne] = reponses.map(codage)

In [105]:
# Une valeur non reconnue est affichée en haut
assert dataset_code[style].notna().all().all(), "Vérifier les réponses affichées au-dessus."

dataset_code[style] = dataset_code[style].astype(int)
dataset_code.head()

,Q5_1,Q5_2,Q5_3,Q5_4,Q5_5,Q5_6,Q5_7,Q5_8,Q5_9,Q5_10,Q5_11,Q5_12,Q5_13,REG13
0,1,0,0,0,1,0,0,0,0,1,0,0,0,LanguedicRoussillonMidiP
1,0,1,0,0,1,0,0,0,0,0,0,1,0,PaysDeLaLoire1
2,1,1,1,0,0,0,0,0,1,0,0,0,0,NordPasDeCalaisPicardie
3,1,0,0,1,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,AuvergneRhoneAlpes


## noms aux colonnes
Les noms des styles

In [106]:
noms_colonnes = {
    "Q5_1": "variete_francaise",
    "Q5_2": "pop_rock",
    "Q5_3": "rap_hip_hop",
    "Q5_4": "classique",
    "Q5_5": "jazz",
    "Q5_6": "dance",
    "Q5_7": "electronique",
    "Q5_8": "metal",
    "Q5_9": "rnb",
    "Q5_10": "soul",
    "Q5_11": "reggae",
    "Q5_12": "musiques_du_monde",
    "Q5_13": "autres_genres",
    "REG13": "region"
}

dataset = dataset_code.rename(columns=noms_colonnes).reset_index(drop=True)
dataset.head()

,variete_francaise,pop_rock,rap_hip_hop,classique,jazz,dance,electronique,metal,rnb,soul,reggae,musiques_du_monde,autres_genres,region
0,1,0,0,0,1,0,0,0,0,1,0,0,0,LanguedicRoussillonMidiP
1,0,1,0,0,1,0,0,0,0,0,0,1,0,PaysDeLaLoire1
2,1,1,1,0,0,0,0,0,1,0,0,0,0,NordPasDeCalaisPicardie
3,1,0,0,1,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,AuvergneRhoneAlpes


### Des styles qui disent la même chose ?

Certains styles vont souvent ensemble : une personne qui écoute de la Soul écoute peut-être aussi du R&B. Si deux colonnes sont presque toujours identiques, elles apportent la même information deux fois.

On calcule la **corrélation** entre chaque paire de styles. Comme les colonnes valent 0 ou 1, cette corrélation s'appelle le coefficient **phi**. Elle va de -1 à 1 :

- proche de **1** : les deux styles sont presque toujours écoutés ensemble ;
- proche de **0** : aucun lien entre les deux ;
- **négative** : quand on écoute l'un, on a tendance à ne pas écouter l'autre.

On affiche les 10 paires les plus liées, puis celles au-dessus d'un seuil de 0,6.

In [107]:
genres = [noms_colonnes[c] for c in style]

# Sur des colonnes 0/1, la corrélation de Pearson correspond au coefficient phi
correlations = dataset[genres].corr()

# On garde chaque paire une seule fois (triangle supérieur)
masque = np.triu(np.ones(correlations.shape, dtype=bool), k=1)
paires = (correlations.where(masque)
          .stack()
          .sort_values(ascending=False)
          .rename("correlation"))

print("Les 10 paires de styles les plus liées :")
display(paires.head(10).round(2))

seuil = 0.6
fortes = paires[paires.abs() >= seuil]
print(f"Paires au-dessus de {seuil} :", len(fortes))
display(fortes.round(2))

Les 10 paires de styles les plus liées :


rap_hip_hop  rnb                  0.46
soul         reggae               0.31
classique    jazz                 0.30
jazz         soul                 0.28
dance        electronique         0.24
             rnb                  0.24
classique    musiques_du_monde    0.23
rnb          soul                 0.23
             reggae               0.20
jazz         musiques_du_monde    0.19
Name: correlation, dtype: float64

Paires au-dessus de 0.6 : 0


Series([], Name: correlation, dtype: float64)

- Les **10 premières paires** montrent quels styles vont le plus souvent ensemble.
- Une corrélation **au-dessus de 0,6** signale une forte redondance : on pourrait envisager de fusionner les deux styles.
- Entre **0,3 et 0,6** : lien modéré, les deux styles apportent encore chacun une info propre. On les garde.
- **En dessous de 0,3** : lien faible, rien à faire.

Ici nous avons 0 paire la plus corrélée et sa valeur, nombre de paires au-dessus de 0,6

In [108]:
noms_regions = {
    "IleDeFrance1": "Île-de-France",
    "AuvergneRhoneAlpes": "Auvergne-Rhône-Alpes",
    "AlsaceChampagneArdenneLo": "Grand Est",
    "AquitaineLimousinPoitouC": "Nouvelle-Aquitaine",
    "NordPasDeCalaisPicardie": "Hauts-de-France",
    "LanguedicRoussillonMidiP": "Occitanie",
    "ProvenceAlpesCoteDAzur": "Provence-Alpes-Côte d'Azur",
    "PaysDeLaLoire1": "Pays de la Loire",
    "Bretagne1": "Bretagne",
    "Normandie": "Normandie",
    "BourgogneFrancheComte": "Bourgogne-Franche-Comté",
    "CentreValDeLoire": "Centre-Val de Loire"
}

dataset["region"] = dataset["region"].replace(noms_regions)
dataset["region"].value_counts()

region
Île-de-France                 404
Auvergne-Rhône-Alpes          237
Grand Est                     200
Nouvelle-Aquitaine            194
Hauts-de-France               188
Occitanie                     161
Provence-Alpes-Côte d'Azur    129
Pays de la Loire              125
Bretagne                       97
Normandie                      89
Bourgogne-Franche-Comté        84
Centre-Val de Loire            78
Name: count, dtype: int64

## Dataset final

En plus des premiers contrôles (pas de valeur manquante, 14 colonnes), on vérifie qu'aucune colonne de localisation de l'étape 1 ne reste dans le dataset final.

Si c'est le cas, la cellule s'arrête avec un message d'erreur qui indique la colonne en cause. Si la cellule passe sans erreur, le dataset ne contient que les 13 styles et la région à prédire.

In [109]:
controle = pd.DataFrame({
    "Type": dataset.dtypes,
    "Valeurs manquantes": dataset.isna().sum(),
    "Valeurs distinctes": dataset.nunique()
})

display(controle)

assert dataset.isna().sum().sum() == 0, "Il reste des valeurs manquantes."
assert dataset.shape[1] == 14, "On attend 13 préférences et une région."
#-------------------------controle automatiqque des fuites-------------------------
fuites = set(colonnes_localisation) & set(dataset.columns)
assert not fuites, f"Colonnes de localisation présentes : {fuites}"


print("Dimensions finales :", dataset.shape)
print("Nombre de régions :", dataset["region"].nunique())

,Type,Valeurs manquantes,Valeurs distinctes
variete_francaise,int64,0,2
pop_rock,int64,0,2
rap_hip_hop,int64,0,2
classique,int64,0,2
jazz,int64,0,2
dance,int64,0,2
electronique,int64,0,2
metal,int64,0,2
rnb,int64,0,2
soul,int64,0,2


Dimensions finales : (1986, 14)
Nombre de régions : 12


In [110]:
repartition_regions = pd.DataFrame({
    "Personnes": dataset["region"].value_counts(),
    "Part (%)": (dataset["region"].value_counts(normalize=True) * 100).round(1)
})

display(repartition_regions)

,Personnes,Part (%)
region,,
Île-de-France,404,20.3
Auvergne-Rhône-Alpes,237,11.9
Grand Est,200,10.1
Nouvelle-Aquitaine,194,9.8
Hauts-de-France,188,9.5
Occitanie,161,8.1
Provence-Alpes-Côte d'Azur,129,6.5
Pays de la Loire,125,6.3
Bretagne,97,4.9


Le fichier contient 12 régions. 
Pzr contre, les classes n’ont pas toutes le même effecti

In [111]:
print("Lignes identiques sur les préférences et la région :", dataset.duplicated().sum())

Lignes identiques sur les préférences et la région : 638


Plusieurs personnes peuvent aimer les mêmes style et vivre dans la même région, les 638 n'est pas u doublons

### Une fonction de nettoyage commune

Tout le nettoyage est réparti dans plusieurs cellules. Pour qu'on puisse tous les 4 partir du même dataset, on rassemble toutes les étapes dans une seule fonction : `nettoyer(df)`.

Elle enchaîne :
1. la sélection des 13 styles et de la région ;
2. le traitement des valeurs manquantes (méthode choisie à l'étape 2) ;
3. le codage des styles en 0/1 ;
4. le renommage des colonnes et des régions ;
5. les contrôles (pas de manque, 14 colonnes, pas de fuite).

Ensuite, on vérifie que la fonction donne **exactement** le même résultat que le notebook cellule par cellule.

In [112]:
def nettoyer(df_brut, methode_manques="suppression"):
    """Reproduit tout le nettoyage : sélection, manques, codage, renommage, contrôles."""
    style = [f"Q5_{i}" for i in range(1, 14)]
    colonnes_utiles = style + ["REG13"]

    # 1. Sélection et code d'absence
    donnees = df_brut[colonnes_utiles].copy().replace("#NUL!", pd.NA)

    # 2. Valeurs manquantes
    donnees = appliquer_manques(donnees, style, colonnes_utiles, methode_manques)

    # 3. Codage 0/1 des styles
    for colonne in style:
        reponses = donnees[colonne].astype("string").str.strip()
        donnees[colonne] = reponses.map(codage)
    if donnees[style].isna().any().any():
        raise ValueError("Réponses non reconnues dans les styles : vérifier le dictionnaire codage.")
    donnees[style] = donnees[style].astype(int)

    # 4. Renommage des colonnes et des régions
    donnees = donnees.rename(columns=noms_colonnes)
    donnees["region"] = donnees["region"].replace(noms_regions)

    # 5. Contrôles
    assert donnees.isna().sum().sum() == 0, "Il reste des valeurs manquantes."
    assert donnees.shape[1] == 14, "On attend 13 préférences et une région."
    assert not set(colonnes_localisation) & set(donnees.columns), "Fuite de localisation."

    return donnees.reset_index(drop=True)


# Vérification : la fonction doit donner exactement le même résultat que le notebook
dataset_fonction = nettoyer(df, METHODE_MANQUES)
pd.testing.assert_frame_equal(dataset_fonction, dataset.reset_index(drop=True))
print("La fonction reproduit le notebook à l'identique :", dataset_fonction.shape)

La fonction reproduit le notebook à l'identique : (1986, 14)


On gère deux cas possible soit la fonction nous renvoie "notebook identique" soit une erreur `AssertionError` apparaît : une étape diffère entre la fonction et le notebook. Le message indique la colonne ou la valeur en cause.

## Préparer les entrées et la cible
`X` contient les 13 styles de musique

`y`  la région à prédire. 

(La région n’entre pas dans `X`, pour éviter de donner la réponse au modèle)

In [113]:
X = dataset.drop(columns="region")
y = dataset["region"]

print("Préférences X :", X.shape)
print("Région y :", y.shape)

Préférences X : (1986, 13)
Région y : (1986,)


##  new dataset
Le CSV contient les styles `0/1` et la cible `region`

In [114]:
from pathlib import Path

Path("../Dataset").mkdir(exist_ok=True)
dataset.to_csv("../Dataset/dataset_musique_region.csv", index=False, encoding="utf-8")

In [115]:
# Vérifier que le fichier enregistré contient exactement les mêmes données.
verification = pd.read_csv("../Dataset/dataset_musique_region.csv")
assert verification.equals(dataset), "Le fichier enregistré est different du dataset."

print("CSV enregistré :", verification.shape)
verification.head()

CSV enregistré : (1986, 14)


,variete_francaise,pop_rock,rap_hip_hop,classique,jazz,dance,electronique,metal,rnb,soul,reggae,musiques_du_monde,autres_genres,region
0,1,0,0,0,1,0,0,0,0,1,0,0,0,Occitanie
1,0,1,0,0,1,0,0,0,0,0,0,1,0,Pays de la Loire
2,1,1,1,0,0,0,0,0,1,0,0,0,0,Hauts-de-France
3,1,0,0,1,0,0,0,0,0,0,0,0,0,Provence-Alpes-Côte d'Azur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,Auvergne-Rhône-Alpes


**Résultat : 1 986 personnes, 13 préférences musicales et 1 région cible.**

In [117]:
# cellule de verifications temporaire 
print("Méthode choisie :", METHODE_MANQUES)
print("Avec suppression :", len(methode_a))
print("Avec remplacement :", len(methode_b))
print("Lignes dans le CSV :", len(verification))

Méthode choisie : suppression
Avec suppression : 1986
Avec remplacement : 1986
Lignes dans le CSV : 1986
